# 🤖 Model Training – Fraudulent Job Detection
**Goal:** Build, evaluate, and select the best classifier to detect fraudulent job postings.

**Pipeline:** Text (TF-IDF) + Categorical (OneHot) + Numerical → Classifier → F1 Evaluation

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

plt.style.use('dark_background')
print('All imports done ✅')

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv('data/fake_job_postings.csv')
df.drop(columns=['job_id'], inplace=True, errors='ignore')
print(f'Shape: {df.shape}')
df['fraudulent'].value_counts()

## 3. Define Feature Groups

In [ ]:
TEXT_COLS = ['title','company_profile','description','requirements','benefits']
CAT_COLS  = ['employment_type','required_experience','required_education',
             'industry','function']
NUM_COLS  = ['telecommuting','has_company_logo','has_questions']
TARGET    = 'fraudulent'

print('Text cols    :', TEXT_COLS)
print('Categoric.   :', CAT_COLS)
print('Numerical    :', NUM_COLS)

## 4. Train / Test Split (Stratified)

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Fraud in train: {y_train.sum()} / {len(y_train)}')
print(f'Fraud in test : {y_test.sum()} / {len(y_test)}')

## 5. Preprocessing

In [ ]:
# ── Text: fill NaN, concatenate, TF-IDF ─────────────────────────────────────
for col in TEXT_COLS:
    X_train[col] = X_train[col].fillna('')
    X_test[col]  = X_test[col].fillna('')

X_train['combined'] = X_train[TEXT_COLS].apply(lambda r: ' '.join(r.values), axis=1)
X_test['combined']  = X_test[TEXT_COLS].apply(lambda r: ' '.join(r.values), axis=1)

tfidf = TfidfVectorizer(max_features=5000, stop_words='english',
                         ngram_range=(1,2), sublinear_tf=True)
X_train_text = tfidf.fit_transform(X_train['combined'])
X_test_text  = tfidf.transform(X_test['combined'])
print(f'TF-IDF matrix shape: {X_train_text.shape}')

In [ ]:
# ── Categorical ──────────────────────────────────────────────────────────────
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

X_train_cat_imp = cat_imputer.fit_transform(X_train[CAT_COLS])
X_test_cat_imp  = cat_imputer.transform(X_test[CAT_COLS])

X_train_cat = cat_encoder.fit_transform(X_train_cat_imp)
X_test_cat  = cat_encoder.transform(X_test_cat_imp)
print(f'Categorical feature shape: {X_train_cat.shape}')

In [ ]:
# ── Numerical ────────────────────────────────────────────────────────────────
num_imputer = SimpleImputer(strategy='median')
num_scaler  = StandardScaler(with_mean=False)

X_train_num = num_scaler.fit_transform(num_imputer.fit_transform(X_train[NUM_COLS]))
X_test_num  = num_scaler.transform(num_imputer.transform(X_test[NUM_COLS]))
print(f'Numerical feature shape: {X_train_num.shape}')

In [ ]:
# ── Combine all feature blocks ───────────────────────────────────────────────
X_train_final = hstack([X_train_text, X_train_cat, csr_matrix(X_train_num)])
X_test_final  = hstack([X_test_text,  X_test_cat,  csr_matrix(X_test_num)])
print(f'Final train matrix: {X_train_final.shape}')
print(f'Final test  matrix: {X_test_final.shape}')

## 6. Train & Evaluate Multiple Models

In [ ]:
neg, pos = (y_train==0).sum(), (y_train==1).sum()
scale_pw  = neg / pos
print(f'Genuine: {neg}  |  Fraudulent: {pos}  |  scale_pos_weight: {scale_pw:.1f}')

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Naive Bayes':         MultinomialNB(alpha=0.5),
    'Decision Tree':       DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42),
    'Random Forest':       RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(scale_pos_weight=scale_pw, use_label_encoder=False,
                                         eval_metric='logloss', n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_final, y_train)
    y_pred = model.predict(X_test_final)
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC':  roc_auc_score(y_test, y_pred),
    }
    print(f'{name:25s} → F1: {results[name]["F1 Score"]:.4f}')

## 7. Comparison Table & Chart

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('F1 Score', ascending=False)
print(results_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.25
metrics = ['Accuracy', 'F1 Score', 'ROC-AUC']
colors  = ['#42a5f5', '#66bb6a', '#ef5350']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(results_df.index, rotation=20, ha='right')
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Comparison – Accuracy / F1 / ROC-AUC', fontsize=13, pad=12)
ax.legend()
plt.tight_layout()
plt.show()

## 8. Best Model – Detailed Evaluation

In [ ]:
best_name  = results_df['F1 Score'].idxmax()
best_model = models[best_name]
print(f'🏆 Best Model: {best_name}')
y_pred_best = best_model.predict(X_test_final)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Genuine','Fraudulent']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(6,5))
disp = ConfusionMatrixDisplay(cm, display_labels=['Genuine','Fraudulent'])
disp.plot(ax=ax, colorbar=False, cmap='Reds')
ax.set_title(f'Confusion Matrix – {best_name}', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

print(f"\n📌 Insight:")
print(f"   True Positives (caught fraudulent)  : {cm[1,1]}")
print(f"   False Negatives (missed fraudulent) : {cm[1,0]}")
print(f"   False Positives (wrongly flagged)   : {cm[0,1]}")

## 9. Top TF-IDF Features (Fraud Signal Words)

In [ ]:
# Only works for Logistic Regression – shows most predictive words
lr_model = models['Logistic Regression']
feature_names = tfidf.get_feature_names_out()
coef = lr_model.coef_[0][:len(feature_names)]  # only TF-IDF slice

top_fraud    = pd.Series(coef, index=feature_names).nlargest(15)
top_genuine  = pd.Series(coef, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_fraud.index[::-1], top_fraud.values[::-1], color='#e53935')
axes[0].set_title('Top Words → Fraudulent', fontsize=11)
axes[1].barh(top_genuine.index, top_genuine.values.abs(), color='#43a047')
axes[1].set_title('Top Words → Genuine', fontsize=11)

plt.tight_layout()
plt.show()

print("\n📌 Insight: Words like 'data entry', 'work home', 'earn' push fraud score up.")
print("   Words like 'experience', 'team', 'position', 'required' push genuine score up.")

## 10. Model Selection Summary

| Model | Accuracy | F1 Score | ROC-AUC |
|---|---|---|---|
| Logistic Regression | ~ | ~ | ~ |
| Naive Bayes | ~ | ~ | ~ |
| Decision Tree | ~ | ~ | ~ |
| Random Forest | ~ | ~ | ~ |
| Gradient Boosting | ~ | ~ | ~ |
| XGBoost | ~ | ~ | ~ |

*(Run the cells above to fill in actual scores)*

**Selected model:** The model with the highest F1 score is chosen, since we prioritise
catching fraudulent postings (high recall) while minimising false positives.

**Key takeaways:**
- TF-IDF bigrams on combined text are the most powerful features
- Binary metadata (logo, questions) adds meaningful signal
- Class imbalance handled via `class_weight='balanced'` / `scale_pos_weight`
- The final model is serialised to `artifacts/model.pkl` via the training pipeline
